# 🧬 PathoShift Challenge — Análise Exploratória de Dados (EDA)
## Entrega 1: Pathology Image Pipeline

Este notebook documenta a exploração das imagens histopatológicas do dataset **Kather Colorectal Histology**.

### Objetivos desta Análise:
1. **Verificação de Classes e Balanceamento:** Confirmar contagem por classe e integridade dimensional.
2. **Inspeção Visual:** Observar a morfologia tecidual típica de cada uma das 8 classes.
3. **Análise de Cor e Espaços Cromáticos:** Comparar perfis nos espaços RGB, HSV e L\*a\*b\*.
4. **Controle de Qualidade:** Identificar tiles com predominância de fundo e quantificar nitidez/foco via Laplaciano.
5. **Validação do Pré-processamento:** Demonstrar o efeito da normalização de coloração (Método de Reinhard).

In [ ]:
import sys
from pathlib import Path

# Adiciona o diretório 'src' ao sys.path para importar o pacote pathoshift
SRC_DIR = Path("..").resolve() / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image

from pathoshift.config import ExperimentConfig
from pathoshift.utils.reproducibility import set_seed
from pathoshift.data.loader import validate_dataset, load_images_and_labels
from pathoshift.preprocessing.quality import detect_background, assess_focus, quality_report
from pathoshift.preprocessing.color_norm import ReinhardNormalizer

# Configurações de estilo
set_seed(42)
sns.set_theme(style="whitegrid", font_scale=1.05)
print("Ambiente e módulos do PathoShift carregados com sucesso!")

## 1. Carregamento e Estatísticas do Dataset

In [ ]:
DATA_DIR = Path("../data/kather_texture_2016_image_tiles_5000")

if not DATA_DIR.exists():
    print(f"Aviso: O diretório {DATA_DIR} ainda não foi descompactado.")
    print("Baixe o dataset do Kaggle (Colorectal Histology MNIST) e coloque na pasta 'data'.")
else:
    stats = validate_dataset(DATA_DIR)
    print(f"Total de imagens: {stats['total_images']}")
    print(f"Dimensões encontradas: {stats['dimensions']}")
    print(f"Formatos: {stats['formats']}")
    print(f"Arquivos corrompidos: {len(stats['corrupted'])}")

## 2. Distribuição das Classes
O dataset Kather é ideal para avaliação inicial por ser perfeitamente balanceado (625 imagens em cada uma das 8 classes).

In [ ]:
if DATA_DIR.exists():
    classes = list(stats['classes'].keys())
    counts = list(stats['classes'].values())

    plt.figure(figsize=(10, 4))
    bars = plt.bar(classes, counts, color=sns.color_palette("Blues_d", len(classes)))
    plt.title("Distribuição de Amostras por Classe de Tecido", fontsize=13, weight='bold')
    plt.xlabel("Classe Histológica")
    plt.ylabel("Quantidade de Imagens")
    plt.xticks(rotation=30, ha="right")
    plt.bar_label(bars, padding=3)
    plt.tight_layout()
    plt.show()

## 3. Galeria Visual de Amostras por Classe
Abaixo visualizamos dois exemplos reais de cada categoria para inspecionar as diferenças estruturais e cromáticas.

In [ ]:
if DATA_DIR.exists():
    fig, axes = plt.subplots(2, 8, figsize=(18, 5))
    for c_idx, c_dir in enumerate(sorted(DATA_DIR.iterdir())):
        if not c_dir.is_dir():
            continue
        imgs = sorted(list(c_dir.glob("*.tif")))
        for row in range(2):
            if row < len(imgs):
                img_rgb = np.array(Image.open(imgs[row]).convert("RGB"))
                axes[row, c_idx].imshow(img_rgb)
            axes[row, c_idx].axis("off")
            if row == 0:
                axes[row, c_idx].set_title(c_dir.name.split("_")[-1], fontsize=10, weight='bold')
    plt.suptitle("Amostras Histológicas por Classe (Kather)", fontsize=14, weight='bold')
    plt.tight_layout()
    plt.show()

## 4. Análise de Cor: RGB vs. HSV
- **Espaço RGB:** Os canais Vermelho, Verde e Azul são altamente correlacionados entre si.
- **Espaço HSV:** O canal **Hue** isola o comprimento de onda tonal, permitindo separar as tonalidades roxas da Hematoxilina (DNA/núcleos celulares) dos tons rosados da Eosina (citoplasma e proteínas da matriz).

In [ ]:
if DATA_DIR.exists():
    sample_path = list(DATA_DIR.rglob("*.tif"))[0]
    sample_rgb = np.array(Image.open(sample_path).convert("RGB"))
    sample_hsv = cv2.cvtColor(sample_rgb, cv2.COLOR_RGB2HSV)

    fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
    axes[0].imshow(sample_rgb)
    axes[0].set_title("Original (RGB)")
    axes[0].axis("off")

    axes[1].imshow(sample_hsv[:, :, 0], cmap='twilight')
    axes[1].set_title("Canal Hue (Matiz)")
    axes[1].axis("off")

    axes[2].imshow(sample_hsv[:, :, 1], cmap='plasma')
    axes[2].set_title("Canal Saturation (Saturação)")
    axes[2].axis("off")

    axes[3].imshow(sample_hsv[:, :, 2], cmap='gray')
    axes[3].set_title("Canal Value (Brilho)")
    axes[3].axis("off")
    plt.tight_layout()
    plt.show()

## 5. Controle de Qualidade: Detecção de Fundo e Medição de Foco
Testamos a função `detect_background` e `assess_focus` em duas amostras:

In [ ]:
if DATA_DIR.exists():
    # Amostra com tecido
    tissue_sample = np.array(Image.open(list((DATA_DIR / '01_TUMOR').glob('*.tif'))[0]).convert('RGB'))
    # Amostra com fundo vazio
    empty_sample = np.array(Image.open(list((DATA_DIR / '08_EMPTY').glob('*.tif'))[0]).convert('RGB'))

    rep_tissue = quality_report(tissue_sample)
    rep_empty = quality_report(empty_sample)

    print("--- Relatório: Amostra TUMOR ---")
    print(f"É fundo? {rep_tissue['is_background']} (Proporção de branco: {rep_tissue['background_ratio']:.1%})")
    print(f"Variância do Laplaciano (Foco): {rep_tissue['focus_variance']}")

    print("\n--- Relatório: Amostra EMPTY ---")
    print(f"É fundo? {rep_empty['is_background']} (Proporção de branco: {rep_empty['background_ratio']:.1%})")
    print(f"Variância do Laplaciano (Foco): {rep_empty['focus_variance']}")

## 6. Normalização de Cor (Reinhard)
Abaixo demonstramos o método de Reinhard no espaço L\*a\*b\*, transferindo a distribuição estatística de uma amostra de referência para uma imagem com coloração divergente.

In [ ]:
if DATA_DIR.exists():
    tumor_imgs = list((DATA_DIR / '01_TUMOR').glob('*.tif'))
    ref_img = np.array(Image.open(tumor_imgs[0]).convert('RGB'))
    source_img = np.array(Image.open(tumor_imgs[10]).convert('RGB'))

    normalizer = ReinhardNormalizer().fit(ref_img)
    normalized_img = normalizer.transform(source_img)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(ref_img)
    axes[0].set_title("Imagem de Referência (Alvo)")
    axes[0].axis("off")

    axes[1].imshow(source_img)
    axes[1].set_title("Imagem Fonte (Original)")
    axes[1].axis("off")

    axes[2].imshow(normalized_img)
    axes[2].set_title("Imagem Normalizada (Reinhard)")
    axes[2].axis("off")
    plt.tight_layout()
    plt.show()